In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Data Classification**

## **Importing Libraries**

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import learning_curve
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## **Data Loading & Inspection**

### **Reading Train Data**

In [ ]:
House_train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')

In [ ]:
House_train.head()

In [ ]:
House_train.info()

In [ ]:
House_train.shape

In [ ]:
House_train.describe()

In [ ]:
House_train.columns

### **Nulls for Train data**

In [ ]:
House_train.isnull().sum()

To extract all columns which have Nulls :

In [ ]:
for col in House_train.columns:
    if House_train[col].isnull().sum() != 0:
        print(f'{col} contain : ({House_train[col].isnull().sum()}) Null')

To keep Id columns found becaue we need it for submission file:

In [ ]:
Train_Id = House_train['Id'].copy()

To remove non important columns and columns which have more than 85% Nulls:

In [ ]:
House_train = House_train.drop(['MiscFeature','Fence','PoolQC','Alley','Id'],axis=1)

In [ ]:
for col in House_train.columns:
    if House_train[col].isnull().sum() != 0:
        print(f'{col} contain : ({House_train[col].isnull().sum()}) Null')

To remove Nulls at Numerical columns we use this code:

In [ ]:
numeric_cols = House_train.select_dtypes(include=['int64','float64']).columns.tolist()

for col in numeric_cols:
    if col in House_train.columns:
        House_train[col].fillna(House_train[col].median(), inplace=True)

To remove Nulls at Categorical columns we use this code:

In [ ]:
categorical_cols = ['MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1',
                    'BsmtFinType2','Electrical','FireplaceQu','GarageType','GarageFinish',
                    'GarageQual']

for col in categorical_cols:
    if col in House_train.columns:
        # fill Nulls by mode
        most_frequent = House_train[col].mode()[0]
        House_train[col].fillna(most_frequent, inplace=True)

In [ ]:
House_train['GarageCond'] = House_train['GarageCond'].astype(str).str.strip()

In [ ]:
# replace nan by None
House_train['GarageCond'].replace('nan', 'None', inplace=True)

To check if there is found any Nulls at data:

In [ ]:
Nulls_col = House_train.isnull().sum()
if Nulls_col.any():
    print(Nulls_col[Nulls_col != 0])
else:
    print('No Nulls Found')

### **Outliers for Train data**

Visualization by Boxplot to all Numerical Columns:

In [ ]:
Numerical_columns = House_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

House_train[Numerical_columns].boxplot(figsize=(15,8), rot=45)
plt.title("Boxplot of Numerical Columns")
plt.show()

To show outliers by using Boxplot of **LotFrontage**:

In [ ]:
plt.boxplot(House_train['LotFrontage'])
plt.axis = ['equal']

To remove Outliers by using **IQR**:

In [ ]:
x = 'LotFrontage'
q75,q25 = np.percentile(House_train.loc[:,x],[75,25])
IQR = q75 - q25
Upper = q75 + (1.4  * IQR)
Lower = q25 - (1.5 *  IQR)
House_train.loc[House_train[x] < Lower,x ]= np.nan
House_train.loc[House_train[x] > Upper,x ]= np.nan

To rerplace Nan Outliers to median:

In [ ]:
House_train['LotFrontage'].fillna(House_train['LotFrontage'].median(), inplace=True)

To show Boxplot to **LotFrontage** after removing Outliers:

In [ ]:
plt.boxplot(House_train['LotFrontage'])
plt.axis = ['equal']

To show outliers by using Boxplot of **SalePrice**:

In [ ]:
plt.boxplot(House_train['SalePrice'])
plt.axis = ['equal']

To remove Outliers by using **IQR:**

In [ ]:
x = 'SalePrice'
q75,q25 = np.percentile(House_train.loc[:,x],[75,25])
IQR = q75 - q25
Upper = q75 + (3.5  * IQR)
Lower = q25 - (1.5 *  IQR)
House_train.loc[House_train[x] < Lower,x ]= np.nan
House_train.loc[House_train[x] > Upper,x ]= np.nan

To rerplace Nan Outliers to median:

In [ ]:
House_train['SalePrice'].fillna(House_train['SalePrice'].median(), inplace=True)


To show Boxplot to **SalePrice** after removing Outliers:

In [ ]:
plt.boxplot(House_train['SalePrice'])
plt.axis = ['equal']

## **FeatureExtraction for Train data**

F**or Numerical values:**

In [ ]:
Numerical_columns = House_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [ ]:
print(Numerical_columns)

For Extraxt **Categorical columns:**

In [ ]:
Categorical_columns = House_train.select_dtypes(include=['object']).columns.tolist()

In [ ]:
print(Categorical_columns)

For extract unique values at evry feature:

In [ ]:
for col in Categorical_columns:
    unique_values = House_train[col].unique()
    print(f'{col} ({len(unique_values)} values): {unique_values}')
    print('-'*50)

We use **LabelEncoder** for these features which have two unique values:

In [ ]:
Lable_encoder = LabelEncoder()
Binary_cols = ['Street', 'CentralAir']
# Apply labelencoder
for col in Binary_cols:
    House_train[col] = Lable_encoder.fit_transform(House_train[col])

We use **OneHotEncoder** for this Features which have Low Cardinality:

In [ ]:
onehot_cols = [
    'MSZoning','LotShape','LandContour','Utilities','LotConfig','LandSlope',
    'BldgType','HouseStyle','RoofStyle','RoofMatl','MasVnrType','Foundation',
    'Heating','Electrical','Functional','GarageFinish','PavedDrive','SaleType','SaleCondition']

existing_onehot_cols = [col for col in onehot_cols if col in House_train.columns]
House_train = pd.get_dummies(House_train, columns=existing_onehot_cols, drop_first=True)

We use **Ordinal Encoding** for these features which have arranged values:

In [ ]:
ordinal_cols = {
    'ExterQual': {'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'ExterCond': {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'BsmtQual': {'None':0,'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'BsmtCond': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4},
    'BsmtExposure': {'None':0,'No':1,'Mn':2,'Av':3,'Gd':4},
    'BsmtFinType1': {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6},
    'BsmtFinType2': {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6},
    'FireplaceQu': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'GarageQual': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'GarageCond': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4},
    'HeatingQC': {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'KitchenQual': {'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'GarageType': {'None':0, 'Basment':1 ,'2Types':2,'Detchd':3,'Attchd':4,'BuiltIn':5,'CarPort':6}
}

for col, mapping in ordinal_cols.items():
    if col in House_train.columns:
        House_train[col] = House_train[col].map(mapping)

Replace nan by mode:

In [ ]:
House_train['GarageCond'].fillna(House_train['GarageCond'].mode()[0], inplace=True)

We use **Frequency Encoding** at these features which have large unique values:

In [ ]:
high_card_cols = ['Neighborhood','Condition1','Condition2','Exterior1st','Exterior2nd']

for col in high_card_cols:
    if col in House_train.columns:
        freq = House_train[col].value_counts() / len(House_train)
        House_train[col] = House_train[col].map(freq)

In [ ]:
House_train.shape

## **Splitting Data**

In [ ]:
X = House_train.drop(['SalePrice'],axis=1)
y = House_train['SalePrice']

In [ ]:
#Splitting data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=44, shuffle =True)

#Splitted Data
print('X_train shape is ' , X_train.shape)
print('X_test shape is ' , X_test.shape)
print('y_train shape is ' , y_train.shape)
print('y_test shape is ' , y_test.shape)

## **Modeling**

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    random_state=42)
xgb_model.fit(X, y)
y_pred = xgb_model.predict(X_test)

In [ ]:
# Predictions
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

In [ ]:
y_test[:10]

In [ ]:
y_pred[:10]

## **Evaluation**

In [ ]:
# RMSE
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

# R2 Score
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# Adjusted R2
n_train = len(y_train)
n_test = len(y_test)
p = X_train.shape[1]

adj_r2_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - p - 1)
adj_r2_test = 1 - (1 - r2_test) * (n_test - 1) / (n_test - p - 1)

print(f"Train RMSE: {rmse_train}")
print(f"Test RMSE: {rmse_test}")
print(f"Train R2: {r2_train}")
print(f"Test R2: {r2_test}")
print(f"Train Adjusted R2: {adj_r2_train:.4f}")
print(f"Test Adjusted R2: {adj_r2_test:.4f}")

### **Visualization**

Visualizing Learning Curve for XGBoost Regressor to detect overfitting/underfitting:

In [ ]:
# Generate learning curve data
train_sizes, train_scores, test_scores = learning_curve(
    xgb_model, X, y, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1)

# mean and standard deviation to scores
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

# Plotting
plt.figure(figsize=(10,6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Train R²')
plt.plot(train_sizes, test_mean, 'o-', color='red', label='CV R²')

plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, color='blue', alpha=0.1)
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, color='red', alpha=0.1)

plt.title('Learning Curve for XGBoost Regressor')
plt.xlabel('Training Samples')
plt.ylabel('R² Score')
plt.legend(loc='best')
plt.grid(True)
plt.show()

Scatter plot: Actual vs Predicted:

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual SalePrice')
plt.ylabel('Predicted SalePrice')
plt.title('Actual vs Predicted SalePrice')
plt.show()

### **Reading Test Data**

In [ ]:
House_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
House_test.head()

In [ ]:
House_test.info()

In [ ]:
House_test.shape

In [ ]:
House_test.describe()

In [ ]:
House_test.columns

### **Nulls Test data**

In [ ]:
House_test.isnull().sum()

To extract all columns which have Nulls :

In [ ]:
for col in House_test.columns:
    if House_test[col].isnull().sum() != 0:
        print(f'{col} contain : ({House_test[col].isnull().sum()}) Null')

To keep Id columns found becaue we need it for submission file:

In [ ]:
Test_Id = House_test['Id'].copy()

To remove non important columns and columns which have more than 85% Nulls:

In [ ]:
House_test = House_test.drop(['MiscFeature','Fence','PoolQC','Alley','Id'],axis=1)

In [ ]:
for col in House_test.columns:
    if House_test[col].isnull().sum() != 0:
        print(f'{col} contain : ({House_test[col].isnull().sum()}) Null')

For **Numerical values**:

In [ ]:
Numerical_columns = House_test.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(Numerical_columns)

For **Categorical values:**

In [ ]:
Categorical_columns = House_test.select_dtypes(include=['object']).columns.tolist()
print(Categorical_columns)

To remove Nulls at Numerical columns we use this code:

In [ ]:
numeric_cols = House_test.select_dtypes(include=['int64','float64']).columns.tolist()

for col in numeric_cols:
    if col in House_test.columns:
        House_test[col].fillna(House_test[col].median(), inplace=True)

To remove Nulls at Categorical columns we use this code:

In [ ]:
categorical_cols = ['MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','MSZoning',
                    'BsmtFinType2','Electrical','FireplaceQu','GarageType','GarageFinish','Utilities',
                    'GarageQual','Exterior1st','Exterior2nd','KitchenQual','Functional','SaleType']

for col in categorical_cols:
    if col in House_test.columns:
        # fill Nulls by mode
        most_frequent = House_test[col].mode()[0]
        House_test[col].fillna(most_frequent, inplace=True)

In [ ]:
House_test['GarageCond'] = House_test['GarageCond'].astype(str).str.strip()

In [ ]:
# replace nan by None
House_test['GarageCond'].replace('nan', 'None', inplace=True)

To check if there is found any Nulls at data:

In [ ]:
Nulls_col = House_test.isnull().sum()
if Nulls_col.any():
    print(Nulls_col[Nulls_col != 0])
else:
    print('No Nulls Found')

## **FeatuerExtraction for Test data**

In [ ]:
for col in Categorical_columns:
    unique_values = House_test[col].unique()
    print(f'{col} ({len(unique_values)} values): {unique_values}')
    print('-'*50)

We use **LabelEncoding (binary columns)** for these features which have two unique values:

In [ ]:
binary_map = {
    'Street': {'Pave': 1, 'Grvl': 0},
    'CentralAir': {'Y': 1, 'N': 0},
    'PavedDrive': {'Y': 2, 'P': 1, 'N': 0}}

for col, mapping in binary_map.items():
    if col in House_test.columns:
        House_test[col] = House_test[col].map(mapping)

We use **One-Hot Encoding (nominal categorical columns)** for this Features which have Low Cardinality:

In [ ]:
onehot_cols = [
    'MSZoning','LotShape','LandContour','Utilities','LotConfig','LandSlope',
    'BldgType','HouseStyle','RoofStyle','RoofMatl','MasVnrType','Foundation',
    'Heating','Electrical','Functional','GarageFinish','PavedDrive','SaleType','SaleCondition']

existing_onehot_cols = [col for col in onehot_cols if col in House_test.columns]
House_test = pd.get_dummies(House_test, columns=existing_onehot_cols, drop_first=[False])

We use **Ordinal Encoding (ordered quality columns)** for these features which have arranged values:

In [ ]:
ordinal_cols = {
    'ExterQual': {'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'ExterCond': {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'BsmtQual': {'None':0,'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'BsmtCond': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4},
    'BsmtExposure': {'None':0,'No':1,'Mn':2,'Av':3,'Gd':4},
    'BsmtFinType1': {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6},
    'BsmtFinType2': {'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6},
    'FireplaceQu': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'GarageCond': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'GarageQual': {'None':0,'Po':1,'Fa':2,'TA':3,'Gd':4},
    'HeatingQC': {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5},
    'KitchenQual': {'Fa':1,'TA':2,'Gd':3,'Ex':4},
    'GarageType': {'None':0, 'Basment':1 ,'2Types':2,'Detchd':3,'Attchd':4,'BuiltIn':5,'CarPort':6}
}

for col, mapping in ordinal_cols.items():
    if col in House_test.columns:
        House_test[col] = House_test[col].map(mapping)

In [ ]:
House_test['GarageCond'].fillna(House_test['GarageCond'].mode()[0], inplace=True)

We use **Frequency Encoding** at these features which have large unique values:

In [ ]:
high_card_cols = ['Neighborhood','Condition1','Condition2','Exterior1st','Exterior2nd']

for col in high_card_cols:
    if col in House_test.columns:
        freq = House_test[col].value_counts() / len(House_train)
        House_test[col] = House_test[col].map(freq)

In [ ]:
House_test.shape

## **Prediction**

In [ ]:
House_test = House_test.reindex(columns = X.columns ,fill_value=0)

In [ ]:
Predictions = xgb_model.predict(House_test)

## **Submission**

In [ ]:
#submission = pd.DataFrame({
submission = pd.DataFrame({
    'Id': Test_Id,
    'SalePrice': Predictions})

In [ ]:
print(Predictions[:5])

In [ ]:
submission.to_csv('submission.csv', index = False)